# LLM 05: Pretraining Objectives

Hands-on:
1. Implement causal LM loss from scratch
2. Implement masked LM loss (BERT-style)
3. Implement span-corruption (T5-style)
4. Train a tiny character-level CLM on Shakespeare
5. Exercises: FIM restructuring, comparing objectives

Deps: `pip install torch`

## 1. Causal LM Loss

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)
B, T, V = 2, 6, 100   # batch, seq, vocab

# Simulate model logits
logits = torch.randn(B, T, V)
targets = torch.randint(0, V, (B, T))

# Causal LM: predict token t from tokens <t
# Standard trick: shift inputs and targets
loss = F.cross_entropy(
    logits[:, :-1, :].reshape(-1, V),
    targets[:, 1:].reshape(-1),
)
print(f'causal LM loss: {loss.item():.4f}')
print('(untrained model: should be ~ln(V) =', round(torch.log(torch.tensor(V)).item(), 2), ')')

## 2. Masked LM Loss (BERT-style)

Mask 15% of tokens, predict only those positions.

In [ ]:
MASK_ID = 99   # reserved

def mask_tokens(x, mask_prob=0.15):
    rand = torch.rand(x.shape)
    mask = rand < mask_prob
    labels = x.clone()
    labels[~mask] = -100        # cross_entropy ignores -100
    x_masked = x.clone()
    x_masked[mask] = MASK_ID
    return x_masked, labels

inputs = torch.randint(0, V, (B, T))
masked_inputs, mlm_labels = mask_tokens(inputs)

# Pretend the model ran over masked_inputs and produced logits
logits = torch.randn(B, T, V)

mlm_loss = F.cross_entropy(
    logits.view(-1, V),
    mlm_labels.view(-1),
    ignore_index=-100,
)
print(f'MLM loss: {mlm_loss.item():.4f}')
print(f'fraction masked: {(mlm_labels != -100).float().mean().item():.2f}')

## 3. Span Corruption (T5-style)

Replace contiguous spans with sentinel tokens; target is the corrupted content.

In [ ]:
def span_corrupt(tokens, sentinel_start=95, corrupt_prob=0.15, mean_span=3):
    import random
    src, tgt = [], []
    i, sentinel = 0, sentinel_start
    while i < len(tokens):
        if random.random() < corrupt_prob:
            span_len = max(1, int(random.gauss(mean_span, 1)))
            src.append(sentinel)
            tgt.append(sentinel)
            tgt.extend(tokens[i:i+span_len])
            i += span_len
            sentinel += 1
        else:
            src.append(tokens[i])
            i += 1
    return src, tgt

tokens = list(range(20))
src, tgt = span_corrupt(tokens)
print('input:    ', tokens)
print('corrupted:', src)
print('target:   ', tgt)

## 4. Train a Tiny Character-Level CLM

Use Karpathy's classic: predict the next character of Shakespeare. We'll reuse the Transformer block from session 02.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

text = ('To be, or not to be, that is the question:\n'
        'Whether tis nobler in the mind to suffer\n'
        'The slings and arrows of outrageous fortune,\n'
        'Or to take arms against a sea of troubles\n') * 20

chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
V = len(chars)
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
print(f'vocab size: {V}, data length: {len(data)}')

block_size = 32
batch_size = 16

def get_batch():
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

class TinyGPT(nn.Module):
    def __init__(self, V, d=64, h=4, n=2):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.pos = nn.Embedding(block_size, d)
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(d, h, 4*d, batch_first=True, activation='gelu')
            for _ in range(n)
        ])
        self.head = nn.Linear(d, V)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        h = self.emb(x) + self.pos(pos)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        for blk in self.blocks:
            h = blk(h, src_mask=mask)
        return self.head(h)

torch.manual_seed(0)
model = TinyGPT(V)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)

for step in range(400):
    x, y = get_batch()
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, V), y.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0:
        print(f'step {step:3d}  loss {loss.item():.3f}')

# Sample
ctx = torch.zeros(1, 1, dtype=torch.long)
out = [ctx.item()]
for _ in range(100):
    x = torch.tensor(out[-block_size:]).unsqueeze(0)
    logits = model(x)[:, -1, :]
    nxt = torch.multinomial(F.softmax(logits, dim=-1), 1).item()
    out.append(nxt)
print('\nsample:')
print(''.join(itos[i] for i in out))

## 5. Exercise: Fill-in-the-Middle (FIM) Restructuring

Given a training example `TEXT`, split into `prefix, middle, suffix` and produce:

```
<fim_prefix>{prefix}<fim_suffix>{suffix}<fim_middle>{middle}
```

This is the exact trick that makes code-completion models good at IDE-style gap filling.

**Try it:** write `restructure_fim(text, prefix_frac=0.4, middle_frac=0.2) -> str` and verify round-trip decoding.

## 6. Exercise: Compare Objectives on a Toy Task

Train three tiny models on the same Shakespeare corpus:
1. CLM (as above).
2. MLM (mask 15%, predict masks). Needs bidirectional attention — remove the causal mask.
3. Span corruption.

Then evaluate each on a **held-out completion** task: given 20 chars of prefix, generate the next 20 chars. Qualitatively observe which objective produces more coherent completions. You'll see why CLM dominates for generation.

## 7. Takeaways

- **Pretraining is next-token prediction at scale**, plus variants for different downstream needs.
- **CLM is the winning architecture for generation**; MLM survives for embeddings.
- **Span corruption and FIM are restructuring tricks** that give a decoder more useful patterns.
- **Chinchilla says: ~20 tokens per parameter.** Modern models over-train for inference efficiency.
- **Data quality now matters more than raw scale** at a fixed compute budget.

Next: **06 — Decoding Strategies**, where we control how the trained distribution becomes text.